<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Управление клиентами — вариант задания № 11


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс **Customer** в C#, который будет представлять информацию о клиентах или покупателях. На основе этого класса разработать **2–3 производных класса**, демонстрирующих принципы **наследования и полиморфизма**. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для демонстрации полиморфизма.

### Требования к базовому классу Customer:

**Атрибуты:**

* Идентификатор клиента (**CustomerId**)
* Имя (**Name**)
* Электронная почта (**Email**)

**Методы:**

* **GetFullName()** — метод для получения полного имени клиента.
* **UpdateEmail(string newEmail)** — метод для обновления электронной почты клиента.
* **ViewProfile()** — метод для просмотра профиля клиента.

### Требования к производным классам:

**1. VIPКлиент (VipCustomer):**
Должен содержать дополнительные атрибуты, такие как **Баланс лояльности (LoyaltyPoints)**. Метод **ViewProfile()** должен быть переопределен для отображения дополнительной информации о VIP-клиенте.

**2. ОбычныйКлиент (RegularCustomer):**
Должен содержать дополнительные атрибуты, такие как **Дата регистрации (RegistrationDate)**. Метод **UpdateEmail()** должен быть переопределен для добавления информации о дате последнего обновления электронной почты.

**3. ГрупповойКлиент (GroupCustomer)** *(если требуется третий класс):*
Должен содержать дополнительные атрибуты, такие как **Название группы (GroupName)**. Метод **GetFullName()** должен быть переопределен для отображения названия группы вместо имени клиента.

#### Дополнительное задание
Добавьте к существующим классам конструкторы классов с использованием геттеров и сеттеров и реализуйте взаимодействие объектов между собой.

**Реализованное взаимодействие:**

* VIP-клиент переводит часть своих бонусных баллов другому VIP-клиенту; изменяются оба баланса.
* Групповой клиент добавляет объекты клиентов в группу и вызывает их методы для просмотра профилей. Повторное добавление клиента с тем же ID не допускается.
* Обновление почты обычного клиента отображается и при просмотре участников группы, поскольку группа хранит ссылку на тот же объект.

Конструкторы присваивают значения через свойства с проверками в сеттерах. Полиморфизм демонстрируется при вызове методов через ссылки типа `Customer`.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

public class Customer
{
    private int _customerId;
    private string _name;
    private string _email;

    public int CustomerId
    {
        get { return _customerId; }
        protected set
        {
            if (value <= 0)
                throw new ArgumentException("ID клиента должен быть больше нуля.");
            _customerId = value;
        }
    }

    public string Name
    {
        get { return _name; }
        protected set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Имя не должно быть пустым.");
            _name = value;
        }
    }

    public string Email
    {
        get { return _email; }
        protected set
        {
            // Простая учебная проверка адреса электронной почты.
            if (string.IsNullOrWhiteSpace(value) || value.Contains(" ") ||
                value.IndexOf('@') <= 0 || value.EndsWith("@") ||
                value.IndexOf('@') != value.LastIndexOf('@'))
                throw new ArgumentException("Укажите адрес электронной почты в формате имя@домен.");
            _email = value;
        }
    }

    public Customer(int customerId, string name, string email)
    {
        CustomerId = customerId;
        Name = name;
        Email = email;
    }

    public virtual string GetFullName()
    {
        return Name;
    }

    public virtual void UpdateEmail(string newEmail)
    {
        Email = newEmail;
        Console.WriteLine($"Клиент {GetFullName()}: электронная почта изменена на {Email}.");
    }

    public virtual string ViewProfile()
    {
        return $"ID: {CustomerId} | Имя: {GetFullName()} | Email: {Email}";
    }
}

public class VipCustomer : Customer
{
    private int _loyaltyPoints;

    public int LoyaltyPoints
    {
        get { return _loyaltyPoints; }
        protected set
        {
            if (value < 0)
                throw new ArgumentException("Баланс бонусов не может быть отрицательным.");
            _loyaltyPoints = value;
        }
    }

    public VipCustomer(int customerId, string name, string email, int loyaltyPoints)
        : base(customerId, name, email)
    {
        LoyaltyPoints = loyaltyPoints;
    }

    public void AddLoyaltyPoints(int points)
    {
        if (points <= 0)
            throw new ArgumentException("Количество начисляемых баллов должно быть положительным.");
        LoyaltyPoints = checked(LoyaltyPoints + points);
        Console.WriteLine($"{Name}: начислено {points} баллов.");
    }

    // Взаимодействие: изменяются балансы двух VIP-клиентов.
    public void TransferPointsTo(VipCustomer recipient, int points)
    {
        if (recipient == null)
            throw new ArgumentNullException(nameof(recipient));
        if (ReferenceEquals(this, recipient))
            throw new ArgumentException("Нельзя переводить баллы самому себе.");
        if (points <= 0)
            throw new ArgumentException("Количество переводимых баллов должно быть положительным.");
        if (points > LoyaltyPoints)
        {
            Console.WriteLine($"{Name}: недостаточно баллов для перевода.");
            return;
        }

        // Проверяем переполнение до изменения любого из балансов.
        int recipientBalance = checked(recipient.LoyaltyPoints + points);
        LoyaltyPoints -= points;
        recipient.LoyaltyPoints = recipientBalance;
        Console.WriteLine($"{Name} передал {points} баллов клиенту {recipient.Name}.");
    }

    public override string ViewProfile()
    {
        return $"{base.ViewProfile()} | Баланс лояльности: {LoyaltyPoints}";
    }
}

public class RegularCustomer : Customer
{
    private DateTime _registrationDate;
    private DateTime? _lastEmailUpdate;

    public DateTime RegistrationDate
    {
        get { return _registrationDate; }
        protected set
        {
            if (value.Date > DateTime.Today)
                throw new ArgumentException("Дата регистрации не может быть в будущем.");
            _registrationDate = value.Date;
        }
    }

    public DateTime? LastEmailUpdate
    {
        get { return _lastEmailUpdate; }
        private set { _lastEmailUpdate = value; }
    }

    public RegularCustomer(int customerId, string name, string email, DateTime registrationDate)
        : base(customerId, name, email)
    {
        RegistrationDate = registrationDate;
    }

    public override void UpdateEmail(string newEmail)
    {
        base.UpdateEmail(newEmail);
        LastEmailUpdate = DateTime.Now;
        Console.WriteLine($"Дата обновления почты: {LastEmailUpdate:dd.MM.yyyy HH:mm}.");
    }

    public string GetRegistrationInfo()
    {
        return $"Дата регистрации: {RegistrationDate:dd.MM.yyyy}";
    }

    public override string ViewProfile()
    {
        string update = LastEmailUpdate.HasValue
            ? LastEmailUpdate.Value.ToString("dd.MM.yyyy HH:mm")
            : "ещё не обновлялась";
        return $"{base.ViewProfile()} | {GetRegistrationInfo()} | Обновление почты: {update}";
    }
}

public class GroupCustomer : Customer
{
    private string _groupName;
    private readonly List<Customer> _members = new List<Customer>();

    public string GroupName
    {
        get { return _groupName; }
        protected set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название группы не должно быть пустым.");
            _groupName = value;
        }
    }

    public int MemberCount
    {
        get { return _members.Count; }
    }

    public GroupCustomer(int customerId, string name, string email, string groupName)
        : base(customerId, name, email)
    {
        GroupName = groupName;
    }

    public override string GetFullName()
    {
        return GroupName;
    }

    public void ChangeGroupName(string newGroupName)
    {
        GroupName = newGroupName;
        Console.WriteLine($"Новое название группы: {GroupName}.");
    }

    // Группа хранит ссылки на реальные объекты клиентов.
    public void AddMember(Customer customer)
    {
        if (customer == null)
            throw new ArgumentNullException(nameof(customer));
        if (customer is GroupCustomer)
            throw new ArgumentException("В группу можно добавлять только отдельных клиентов.");
        foreach (Customer member in _members)
        {
            if (member.CustomerId == customer.CustomerId)
            {
                Console.WriteLine($"Клиент {customer.GetFullName()} уже состоит в группе.");
                return;
            }
        }
        _members.Add(customer);
        Console.WriteLine($"Клиент {customer.GetFullName()} добавлен в группу «{GroupName}».");
    }

    public void ShowMembers()
    {
        Console.WriteLine($"Участники группы «{GroupName}»: {MemberCount}");
        foreach (Customer member in _members)
        {
            // Полиморфизм: вызывается реализация конкретного типа клиента.
            Console.WriteLine(member.ViewProfile());
        }
    }
}

// Создание объектов через конструкторы с проверкой свойств.
VipCustomer vip = new VipCustomer(1, "Иван Иванов", "ivan@example.com", 1500);
VipCustomer secondVip = new VipCustomer(2, "Анна Смирнова", "anna@example.com", 300);
RegularCustomer regular = new RegularCustomer(3, "Пётр Петров", "petr@example.com", DateTime.Today.AddDays(-30));
GroupCustomer group = new GroupCustomer(4, "Алексей", "group@example.com", "Студенты");

vip.AddLoyaltyPoints(500);
group.ChangeGroupName("Студенты ТИУ");

Console.WriteLine("\nВзаимодействие объектов");
vip.TransferPointsTo(secondVip, 400);
group.AddMember(vip);
group.AddMember(secondVip);
group.AddMember(regular);

// Обновление объекта видно и при просмотре этого клиента через группу.
Customer customerToUpdate = regular;
customerToUpdate.UpdateEmail("newpetr@example.com");
group.ShowMembers();

Console.WriteLine("\nИтоговые профили: демонстрация полиморфизма");
List<Customer> customers = new List<Customer> { vip, secondVip, regular, group };
foreach (Customer customer in customers)
{
    Console.WriteLine(customer.ViewProfile());
}


The below script needs to be able to find the current output cell; this is an easy method to get it.

Иван Иванов: начислено 500 баллов.
Новое название группы: Студенты ТИУ.

Взаимодействие объектов
Иван Иванов передал 400 баллов клиенту Анна Смирнова.
Клиент Иван Иванов добавлен в группу «Студенты ТИУ».
Клиент Анна Смирнова добавлен в группу «Студенты ТИУ».
Клиент Пётр Петров добавлен в группу «Студенты ТИУ».
Клиент Пётр Петров: электронная почта изменена на newpetr@example.com.
Дата обновления почты: 25.09.2026 21:00.
Участники группы «Студенты ТИУ»: 3
ID: 1 | Имя: Иван Иванов | Email: ivan@example.com | Баланс лояльности: 1600
ID: 2 | Имя: Анна Смирнова | Email: anna@example.com | Баланс лояльности: 700
ID: 3 | Имя: Пётр Петров | Email: newpetr@example.com | Дата регистрации: 26.08.2026 | Обновление почты: 25.09.2026 21:00

Итоговые профили: демонстрация полиморфизма
ID: 1 | Имя: Иван Иванов | Email: ivan@example.com | Баланс лояльности: 1600
ID: 2 | Имя: Анна Смирнова | Email: anna@example.com | Баланс лояльности: 700
ID: 3 | Имя: Пётр Петров | Email: newpetr@example.com | Дата рег